# Hardware-aware synthesis of MCX

Synthesize a ten-control MCX under a depth and width budget. Route the twelve-control MCX on the original 18-qubit grid and star devices. Compare actual CX counts and validate both coherent results.

## Separate logical control from hardware placement

The editable qfunc below specifies an all-ones control over a logical array.
FlagQuantum constructs the bounded MCX decomposition. The linked workflow then
compares explicit placements on the original eighteen-qubit grid and star,
converts to the named basis and verifies coherent states. Those placement
candidates are chosen by host code; the tutorial does not claim that synth
infers the complete search automatically from the device name.

Try another admitted width budget for the same logical operation, then compare
actual CX counts and depths on both topologies. The original zero input never
activates MCX, so retain the active and coherent verification rather than using
its unchanged output as sufficient evidence. CPU routing correctness is distinct
from measured hardware noise or fidelity.


Use FlagQuantum synthesis and the local target compiler.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

import examples.synthesis.community.basic_examples.hw_aware_synthesis.hw_aware_synthesis.overview.main as report
import examples.synthesis.community.basic_examples.hw_aware_synthesis.hw_aware_synthesis.overview.workflow as workflow
from flagquantum import synth
from flagquantum.synth import Output, QArray, QBit, X, allocate, control, qfunc


@qfunc
def mcx(controls: QArray[QBit], target: QBit) -> None:
    control(controls, lambda: X(target))


@qfunc
def main(size: int, controls: Output[QArray[QBit]], target: Output[QBit]) -> None:
    allocate(size, controls)
    allocate(target)
    mcx(controls, target)


preview = synth.synthesize(
    main,
    classical_inputs={"size": 10},
    objective="depth",
    max_qubits=20,
    max_workspace=9,
    basis=("cx", "u"),
    memory_limit_bytes=2**25,
)
print("Logical MCX request:", preview.report["resources"])


Inspect the native multi-control operation.

In [ ]:
mcx = workflow.mcx


Bind the original ten-control instance.

In [ ]:
controls = 10
entry = workflow.main
assert controls == 10


Synthesize the original depth-constrained MCX.

In [ ]:
baseline = workflow.build_program(10)
assert baseline.circuit.n_wires <= 20


Read actual depth and verify the full native result.

In [ ]:
report.report_baseline(baseline)


Reuse the same native MCX for both target topologies.

In [ ]:
assert mcx is workflow.mcx


Bind the original twelve-control hardware case.

In [ ]:
hardware_case = workflow.build_program(12)
assert len(hardware_case.registers["controls"]) == 12


Build both exact 18-qubit coupling maps.

In [ ]:
grid = workflow.coupling_map("grid")
star = workflow.coupling_map("star")
print(grid.n_qubits, len(grid.edges), star.n_qubits, len(star.edges))


Draw the original 3 × 6 grid edges.

In [ ]:
figures = Path("notebook-figures")
report.plot_topology("grid", figures)
display(Image(filename=str(figures / "grid-connectivity.png")))


Draw the original center-to-leaf star links.

In [ ]:
report.plot_topology("star", figures)
display(Image(filename=str(figures / "star-connectivity.png")))


Use the CX and U basis for both devices.

In [ ]:
assert workflow.BASIS == ("cx", "u")


Apply the original 18-qubit ceiling and select on emitted CX.

In [ ]:
assert hardware_case.circuit.n_wires <= 18


Compile the native program for the grid.

In [ ]:
grid_result = workflow.route(hardware_case, "grid")


Print every grid route and verify the chosen result.

In [ ]:
report.report_routing(grid_result)


Compile the same native program for the star.

In [ ]:
star_result = workflow.route(hardware_case, "star")


Print every star route and verify the chosen result.

In [ ]:
report.report_routing(star_result)


Read the selected grid circuit depth.

In [ ]:
print("Grid depth:", grid_result.selected.resources.depth)


Read the selected star circuit depth.

In [ ]:
print("Star depth:", star_result.selected.resources.depth)
